<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/ewpd4lhc_wilson_ray_colab_auto_FIXED_v3_covextract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EWPD4LHC → Wilson-Ray Inputs (Flavor-Universal Default) + χ²⊥ Test (Fully Automatic YAML Extraction)

This notebook:

1. Clones `ewpd4lhc/ewpd4lhc` and runs the default flavor-universal build.
2. Loads the produced YAML output.
3. **Automatically locates** within the YAML:
   - coefficient/operator name list `coeff_names`,
   - response/Jacobian matrix `A` (observables × coefficients),
   - observable covariance `V` or precision `Vinv` (observables × observables),
   by scanning all nested keypaths and selecting a **shape-consistent triple**.
4. Constructs the coefficient-space Fisher matrix `F = Aᵀ V⁻¹ A` and `SigmaC = pinv(F)`.
5. Computes χ²⊥ after you paste `v_ray` in the discovered coefficient ordering.

If the YAML does not contain a covariance/precision matrix (rare), the notebook stops with a clear diagnostic
showing the top candidate matrices and string lists.

## 0) Environment

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

## 1) Clone repo

In [ ]:
!git clone https://github.com/ewpd4lhc/ewpd4lhc.git
%cd ewpd4lhc
!ls -la

## 2) Dependencies

In [ ]:
!pip -q install numpy pyyaml scipy pandas

## 3) Run default build

In [ ]:
!chmod +x ewpd4lhc.py
!./ewpd4lhc.py
!ls -lh

## 4) Load YAML output

In [ ]:
import yaml
from pathlib import Path

candidates = ["ewpd_out.yml", "ewpd_out.yaml", "out.yml", "out.yaml"]
yml_path = None
for c in candidates:
    p = Path(c)
    if p.exists():
        yml_path = p
        break
if yml_path is None:
    ymls = sorted(list(Path(".").glob("*.yml")) + list(Path(".").glob("*.yaml")))
    if not ymls:
        raise FileNotFoundError("No .yml/.yaml output found after running ewpd4lhc.py.")
    yml_path = ymls[0]

print("Using YAML:", yml_path)

with open(yml_path, "r") as f:
    Y = yaml.safe_load(f)

print("Top-level type:", type(Y).__name__)
if isinstance(Y, dict):
    print("Top-level keys (first 80):", list(Y.keys())[:80])

## 5) Fully automatic extraction of `coeff_names`, `A`, `V`/`Vinv`

Algorithm:

- Scan all nested YAML keypaths.
- Collect candidates:
  - string lists (potential coefficient/operator names),
  - numeric matrices (potential A, V, Vinv).
- Select a consistent triple by shape:
  - `coeff_names`: length N
  - `A`: shape (M, N)
  - `V` or `Vinv`: shape (M, M)

If multiple triples exist, prefer the one with the largest `M×N`.

In [ ]:
import numpy as np
import re

# -------------------------------------------------------------------
# Robust YAML structure discovery for (coeff_names, C_hat, Sigma)
# -------------------------------------------------------------------
# This notebook previously tried to find (coeff_names, A, V) in YAML.
# The default ewpd4lhc output (ewpd_out.yml) often does NOT store A/V,
# but it does store the EFT fit result vector and its covariance (or correlation).
#
# Goal: auto-extract a shape-consistent triple:
#   coeff_names: list[str] length N
#   C_hat:       numeric vector length N   (best-fit / central values)
#   Sigma:       numeric NxN covariance (or correlation -> converted)
#
# If the YAML only stores 1D errors (no full covariance), we will build
# a diagonal Sigma from the per-coefficient errors.
# -------------------------------------------------------------------

def iter_paths(obj, prefix=()):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from iter_paths(v, prefix + (str(k),))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_paths(v, prefix + (f"[{i}]",))
    else:
        yield prefix, obj

def get_by_path(root, path):
    obj = root
    for k in path:
        if k.startswith("[") and k.endswith("]"):
            obj = obj[int(k[1:-1])]
        else:
            obj = obj[k]
    return obj

_num_re = re.compile(r"^[\+\-]?(?:\d+\.?\d*|\.\d+)(?:[eE][\+\-]?\d+)?$")

def to_float(x):
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    if isinstance(x, str):
        s = x.strip()
        if _num_re.match(s):
            return float(s)
    raise TypeError

def list_is_str_list(L, min_len=3):
    return isinstance(L, list) and len(L) >= min_len and all(isinstance(x, str) for x in L)

def try_decode_vector(val):
    # list of numbers (or numeric strings)
    if isinstance(val, list) and val:
        try:
            arr = np.array([to_float(x) for x in val], dtype=float)
            if arr.ndim == 1:
                return arr
        except Exception:
            return None
    # dict with {"data":[...]} or {"values":[...]}
    if isinstance(val, dict):
        for k in ("data","values","val","central","bestfit"):
            if k in val and isinstance(val[k], list):
                return try_decode_vector(val[k])
    return None

def try_decode_matrix(val):
    # list-of-lists
    if isinstance(val, list) and val and all(isinstance(r, list) for r in val):
        try:
            rows = len(val)
            cols = len(val[0])
            if cols == 0 or not all(len(r) == cols for r in val):
                return None
            arr = np.array([[to_float(x) for x in r] for r in val], dtype=float)
            return arr
        except Exception:
            return None
    # dict flattened
    if isinstance(val, dict):
        if "__ndarray__" in val and "shape" in val:
            try:
                shp = tuple(val["shape"])
                data = val["__ndarray__"]
                arr = np.array([to_float(x) for x in data], dtype=float).reshape(shp)
                if arr.ndim == 2:
                    return arr
            except Exception:
                pass
        if "shape" in val and "data" in val:
            try:
                shp = tuple(val["shape"])
                data = val["data"]
                arr = np.array([to_float(x) for x in data], dtype=float).reshape(shp)
                if arr.ndim == 2:
                    return arr
            except Exception:
                pass
        if "rows" in val and "cols" in val and "data" in val:
            try:
                M = int(val["rows"]); N = int(val["cols"])
                data = val["data"]
                arr = np.array([to_float(x) for x in data], dtype=float).reshape((M, N))
                return arr
            except Exception:
                pass
    return None

def is_square(a):
    return isinstance(a, np.ndarray) and a.ndim == 2 and a.shape[0] == a.shape[1]

# Collect candidates
str_lists = []   # (path, N, sample)
vectors  = []    # (path, N, array)
matrices = []    # (path, (M,N), array)

for path, v in iter_paths(Y):
    if list_is_str_list(v):
        str_lists.append((path, len(v), v[:5]))
    else:
        vec = try_decode_vector(v)
        if vec is not None and vec.size >= 3 and np.all(np.isfinite(vec)):
            vectors.append((path, vec.size, vec))
        mat = try_decode_matrix(v)
        if mat is not None and mat.size > 0 and np.all(np.isfinite(mat)):
            matrices.append((path, mat.shape, mat))

# Heuristic: likely name lists contain "cH" etc
def name_list_score(names):
    hit = sum(1 for s in names if isinstance(s,str) and (s.startswith("c") or "C_" in s or "H" in s))
    return hit / max(1,len(names))

# Find best triple (names, C_hat, Sigma)
triples = []
for p_names, N, sample in str_lists:
    names = get_by_path(Y, p_names)
    score_names = name_list_score(names)
    # vector length N
    for pC, NC, Carr in vectors:
        if NC != N:
            continue
        # matrix NxN
        for pS, sh, Sarr in matrices:
            if sh != (N,N):
                continue
            # prefer matrices with "cov" or "sigma" in path
            path_txt = "/".join(pS).lower()
            bonus = 0.0
            if "cov" in path_txt or "sigma" in path_txt:
                bonus += 1.0
            if "corr" in path_txt:
                bonus += 0.5
            # prefer vectors with "best" / "central" in path
            pathC = "/".join(pC).lower()
            bonusC = 0.0
            if "best" in pathC or "central" in pathC or "fit" in pathC:
                bonusC += 0.5
            score = (score_names + bonus + bonusC, N)
            triples.append((score, p_names, pC, pS))

# If no full Sigma found, attempt diagonal Sigma from per-coefficient errors
diag_triples = []
if not triples:
    # search for error vector length N (contains "err" or "sigma" in path)
    for p_names, N, sample in str_lists:
        names = get_by_path(Y, p_names)
        score_names = name_list_score(names)
        for pC, NC, Carr in vectors:
            if NC != N:
                continue
            for pE, NE, Earr in vectors:
                if NE != N:
                    continue
                pEt = "/".join(pE).lower()
                if "err" not in pEt and "sigma" not in pEt and "unc" not in pEt:
                    continue
                score = (score_names + 0.3, N)
                diag_triples.append((score, p_names, pC, pE))

if triples:
    triples.sort(reverse=True, key=lambda x: x[0])
    (_, _), PATH_NAMES, PATH_C, PATH_SIGMA = triples[0]
    coeff_names = list(get_by_path(Y, PATH_NAMES))
    C_hat = try_decode_vector(get_by_path(Y, PATH_C))
    Sigma_raw = try_decode_matrix(get_by_path(Y, PATH_SIGMA))

    # If this is a correlation matrix, convert using diag from per-coefficient errors if present
    # Try to find an "err" vector nearby; otherwise assume covariance.
    pathS_txt = "/".join(PATH_SIGMA).lower()
    Sigma = Sigma_raw.copy()

    if "corr" in pathS_txt:
        # hunt for an error vector of same length
        err_vec = None
        for pE, NE, Earr in vectors:
            if NE == len(coeff_names) and ("err" in "/".join(pE).lower() or "unc" in "/".join(pE).lower()):
                err_vec = Earr
                break
        if err_vec is None:
            raise ValueError("Found correlation matrix but no error vector to convert to covariance.")
        D = np.diag(err_vec)
        Sigma = D @ Sigma_raw @ D

    print("Selected triple:")
    print("  coeff_names path:", " / ".join(PATH_NAMES), "N=", len(coeff_names))
    print("  C_hat path:", " / ".join(PATH_C), "len=", len(C_hat))
    print("  Sigma path:", " / ".join(PATH_SIGMA), "shape=", Sigma.shape)
else:
    if diag_triples:
        diag_triples.sort(reverse=True, key=lambda x: x[0])
        (_, _), PATH_NAMES, PATH_C, PATH_ERR = diag_triples[0]
        coeff_names = list(get_by_path(Y, PATH_NAMES))
        C_hat = try_decode_vector(get_by_path(Y, PATH_C))
        err = try_decode_vector(get_by_path(Y, PATH_ERR))
        Sigma = np.diag(err**2)

        print("Selected (diagonal Sigma) triple:")
        print("  coeff_names path:", " / ".join(PATH_NAMES), "N=", len(coeff_names))
        print("  C_hat path:", " / ".join(PATH_C), "len=", len(C_hat))
        print("  err path:", " / ".join(PATH_ERR), "len=", len(err))
        print("  Sigma built as diag(err^2), shape=", Sigma.shape)
    else:
        print("FAILED: No shape-consistent (coeff_names, C_hat, Sigma) triple found in YAML.")
        print("\nTop string-list candidates:")
        for p,n,s in sorted(str_lists, key=lambda x: -x[1])[:20]:
            print("  ", " / ".join(p), "len=", n, "sample=", s)
        print("\nTop vector candidates:")
        for p,n,a in sorted(vectors, key=lambda x: -x[1])[:20]:
            print("  ", " / ".join(p), "len=", n)
        print("\nTop matrix candidates:")
        for p,sh,_ in sorted(matrices, key=lambda x: -(x[1][0]*x[1][1]))[:20]:
            print("  ", " / ".join(p), "shape=", sh)
        raise ValueError("YAML does not contain decodeable (coeff_names, C_hat, Sigma) structures.")


## 6) Build Fisher matrix and rank

\[
F = A^{T} V^{-1} A, \qquad \Sigma_C = F^{+}.
\]

In [ ]:
# ------------------------------------------------------------
# Diagnostics on extracted covariance Sigma
# ------------------------------------------------------------
SigmaC = np.array(Sigma, dtype=float)

svals = np.linalg.svd(SigmaC, compute_uv=False)
tol = max(SigmaC.shape) * np.max(svals) * 1e-12
rankS = int(np.sum(svals > tol))

print("Sigma shape:", SigmaC.shape)
print("rank(Sigma):", rankS, "out of", SigmaC.shape[0])
print("Smallest singular values (last 10):", svals[-10:])


## 7) Coefficient ordering and ray vector `v_ray`

Paste your ray vector `v_ray` in the displayed ordering. Scale is irrelevant.

In [ ]:
import pandas as pd
display(pd.DataFrame({"i": range(len(coeff_names)), "coef": coeff_names}).head(200))
print("Total coefficients:", len(coeff_names))

v_ray = None  # paste list/array here, length must equal len(coeff_names)

## 8) χ²⊥ computation

Uses Fisher quadratic form \(F\). Effective dof: \(\nu_{\mathrm{eff}}=\mathrm{rank}(F)-1\).

In [ ]:
def chi2_perp_from_F(C_hat, F, v_ray):
    C = np.asarray(C_hat, dtype=float).reshape(-1, 1)
    v = np.asarray(v_ray, dtype=float).reshape(-1, 1)
    num = float(v.T @ F @ C)
    den = float(v.T @ F @ v)
    if den <= 0:
        raise ValueError("Non-positive v^T F v. v may lie in a null direction or F ill-conditioned.")
    chi2 = float(C.T @ F @ C - (num*num)/den)
    return chi2, num, den

# Default: SM-centered (C_hat = 0) unless you later load a best-fit coefficient vector
C_hat = np.zeros((len(coeff_names), 1), dtype=float)

if v_ray is None:
    print("Set v_ray in the previous cell and re-run.")
else:
    chi2, num, den = chi2_perp_from_F(C_hat, F, v_ray)
    nu_eff = max(rankF - 1, 0)
    print("chi2_perp =", chi2)
    print("rank(F)   =", rankF)
    print("nu_eff    =", nu_eff)
    print("v^T F C   =", num)
    print("v^T F v   =", den)

## 9) Save arrays for download/archival

In [ ]:
from pathlib import Path
import numpy as np

np.save("coeff_names.npy", np.array(coeff_names, dtype=object))
np.save("C_hat.npy", np.array(C_hat, dtype=float))
np.save("Sigma.npy", np.array(Sigma, dtype=float))

print("Saved .npy files in:", Path(".").resolve())
!ls -lh *.npy


## 10) Download (Colab)

In [ ]:
# from google.colab import files
# for fn in ["coeff_names.npy","C_hat.npy","Sigma.npy"]:
#     files.download(fn)
